## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer

pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = Path('..') / 'data' / 'raw' / 'SciDCC.csv'
assert DATA_PATH.exists(), f'No dataset in: {DATA_PATH.resolve()}'

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)

## 2. Data quality

In [ ]:
# types and missing
print(df.dtypes)
print()
print('Missing:')
print(df.isna().sum())

In [ ]:
# duplicates
print(f'Duplicate on Link:  {df.duplicated(subset=["Link"]).sum()}')
print(f'Duplicate on Title: {df.duplicated(subset=["Title"]).sum()}')
print(f'Full duplicate:     {df.duplicated().sum()}')

In [ ]:
print(f'Godine: {df["Year"].min()}..{df["Year"].max()}')
df['Year'].value_counts().sort_index().tail(10)

Removing rows with empty `Body` (main text) and duplicates on `Link`.

In [ ]:
before = len(df)
df_clean = df.dropna(subset=['Body']).drop_duplicates(subset=['Link']).reset_index(drop=True)
print(f'{before} -> {len(df_clean)}  (uklonjeno {before - len(df_clean)})')

## 3. Class distribution

In [ ]:
class_counts = df_clean['Category'].value_counts()
print(f'No. of classes: {len(class_counts)}')
print(f'Min/max:    {class_counts.min()} / {class_counts.max()}')
print(f'Ratio:      {class_counts.max() / class_counts.min():.1f}x')
print(f'Median:    {class_counts.median():.0f}')
print()
print(class_counts)

In [ ]:
# color per class size
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#c0392b' if c < 50 else '#e67e22' if c < 300 else '#27ae60' for c in class_counts.values]
ax.barh(class_counts.index[::-1], class_counts.values[::-1], color=colors[::-1])
ax.set_xlabel('No. of samples')
ax.set_title('Class distribution\n(red <50, orange <300, green >=300)')
for i, v in enumerate(class_counts.values[::-1]):
    ax.text(v + 10, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
THRESHOLD = 50
small = class_counts[class_counts < THRESHOLD]
print(f'Classes with less than {THRESHOLD} samples:')
print(small)
print(f'\nTotal samples in small classes: {small.sum()} ({small.sum()/len(df_clean)*100:.2f}%)')

## 4. Word count

In [ ]:
for col in ['Title', 'Summary', 'Body']:
    df_clean[f'{col}_words'] = df_clean[col].astype(str).str.split().str.len()

df_clean['Combined_words'] = (
    df_clean['Title'].astype(str) + ' ' +
    df_clean['Summary'].astype(str) + ' ' +
    df_clean['Body'].astype(str)
).str.split().str.len()

df_clean[['Title_words', 'Summary_words', 'Body_words', 'Combined_words']].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['Title_words', 'Summary_words', 'Body_words']):
    data = df_clean[col]
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='red', linestyle='--', label=f'median {data.median():.0f}')
    ax.axvline(data.quantile(0.95), color='orange', linestyle='--', label=f'p95 {data.quantile(0.95):.0f}')
    ax.set_title(col.replace('_words', ''))
    ax.set_xlabel('no. of words')
    ax.legend()
plt.tight_layout()
plt.show()

## 5. BERT Token count

In [ ]:
TOKENIZER_MODEL = 'P0L3/cliscibert_scivocab_uncased'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)
print(f'Vocab size: {tokenizer.vocab_size}')

In [ ]:
def count_tokens(texts, batch_size=64):
    counts = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, add_special_tokens=False, truncation=False)
        counts.extend(len(ids) for ids in enc['input_ids'])
    return counts

df_clean['Title_tokens']   = count_tokens(df_clean['Title'].astype(str))
df_clean['Summary_tokens'] = count_tokens(df_clean['Summary'].astype(str))
print('Body token counting (traje malo)...')
df_clean['Body_tokens']    = count_tokens(df_clean['Body'].astype(str))
print('done')

In [ ]:
df_clean[['Title_tokens', 'Summary_tokens', 'Body_tokens']].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
).round(1)

In [ ]:
# effective limit is 510 (CLS + SEP take 2)
EFFECTIVE_LIMIT = 510
for col in ['Title_tokens', 'Summary_tokens', 'Body_tokens']:
    pct = (df_clean[col] <= EFFECTIVE_LIMIT).mean() * 100
    print(f'{col:18s}  {pct:5.1f}% fit in 510')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_clean['Body_tokens'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(512, color='red', linestyle='--', linewidth=2, label='BERT limit')
ax.axvline(df_clean['Body_tokens'].median(), color='green', linestyle='--', label=f'median {df_clean["Body_tokens"].median():.0f}')
ax.set_xlabel('no. of tokens')
ax.set_title('Body in CliSciBERT tokens')
ax.legend()
plt.tight_layout()
plt.show()

Most Body texts can't fit in 512. Options for truncation:

1. Head (default HuggingFace): first 512.
2. Head + tail: e.g. 256 + 256.
3. Title + Summary, without Body

## 6. Length per class

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
order = df_clean.groupby('Category')['Body_tokens'].median().sort_values().index
sns.boxplot(data=df_clean, y='Category', x='Body_tokens', order=order, ax=ax, showfliers=False)
ax.axvline(512, color='red', linestyle='--', alpha=0.6, label='BERT limit')
ax.set_title('Body length per class')
ax.set_xlabel('no. of tokens')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Distribution through years

In [ ]:
year_category = df_clean.groupby(['Year', 'Category']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(13, 6))
year_category.plot(kind='area', stacked=True, ax=ax, colormap='tab20', linewidth=0)
ax.set_title('Articles per year and category')
ax.set_ylabel('no. of articles')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
yearly = df_clean['Year'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(yearly.index, yearly.values, color='steelblue')
ax.set_title('Total of articles per year')
ax.set_xlabel('year')
plt.tight_layout()
plt.show()

## 8. Examples per class

Manual check: Which classes have semantic overlapping (e.g. `Climate` vs `Global Warming` vs `Weather`).

In [ ]:
for cat in sorted(df_clean['Category'].unique()):
    sub = df_clean[df_clean['Category'] == cat]
    sample = sub['Title'].sample(min(2, len(sub)), random_state=42).tolist()
    print(f'\n[{cat}] ({class_counts[cat]})')
    for s in sample:
        print(f'  * {s}')

## 9. Clean dataset save

In [ ]:
import json

PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

out_path = PROCESSED_DIR / 'scidcc_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Saved: {out_path.resolve()}  {df_clean.shape}')

# label mapping, sorted alphabetically
categories = sorted(df_clean['Category'].unique())
label2id = {cat: i for i, cat in enumerate(categories)}
id2label = {i: cat for i, cat in enumerate(categories)}

map_path = PROCESSED_DIR / 'label_map.json'
with open(map_path, 'w', encoding='utf-8') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, indent=2, ensure_ascii=False)
print(f'Saved: {map_path.resolve()}  ({len(categories)} klasa)')